In [1]:
import emodeconnection as emc
import numpy as np
import matplotlib.pyplot as plt
import time


# Fixed parameters of Geometry
wavelength_fund = 450.0
w_core = 400.0  # [nm] Fixed width
h_core = 350.0  # [nm] Fixed height
dx, dy = 10.0, 10.0


# Sellmeier Material Definition
# Use double asterisks for power, ensure no spaces in the inner math, and verify the negative B3/C3 values are handled cleanly.
eq_o = '(1+2.8032/(1-0.015287/x**2)+0.36335/(1-0.036095/x**2)-33508000/(1+367200000/x**2))**0.5'
eq_e = '(1+0.017061/(1-0.043855/x**2)+3.1976/(1-0.022642/x**2)-57269000/(1-74226000/x**2))**0.5'
anisotropic_equation = f"[{eq_o},{eq_e},{eq_o}]"

In [2]:
# Launch EMode 
em = emc.EMode()



In [3]:
# Set Up Simulation
em.add_material(name='custom_AlN',                                  
                refractive_index_equation=anisotropic_equation, 
                wavelength_unit='um')
em.settings(window_width=2000,window_height=h_core+2000, boundary_condition='TM') # Using your 'TM' preference)
em.shape(name='Substrate', material='Al2O3', height=1000)
em.shape(name='core', material='custom_AlN', height=h_core, mask=w_core, etch_depth=h_core, sidewall_angle=5)
em.shape(name='TopClad', material='SiO2', height=800, shape_type='conformal')
em.plot()


'ok'

In [4]:
# Calculate Modes for FUNDAMENTAL WAVELENGTH
Nmodes_to_calculate = 2
em.settings(wavelength=wavelength_fund, x_resolution=dx, y_resolution=dy,window_width=2000,
            window_height=h_core+2000, num_modes=Nmodes_to_calculate, boundary_condition='TM') # Using your 'TM' preference)
em.FDM()                            #run the finite difference mode solver to find the modes of the structure
em.confinement(shape_list='core')   #calculate the confinement factor for each mode
em.report()                         #print information about the calculation results to command line
em.label_profile(name = 'dataset1') #store this set of results under label '0'
em.plot()


'ok'

In [5]:
# Calculate Modes for SECOND HARMONIC WAVELENGTH
Nmodes_to_calculate = 40
em.settings(num_modes=Nmodes_to_calculate, wavelength=wavelength_fund/2, x_resolution=dx/2, y_resolution=dy/2)
em.FDM()                            #run the finite difference mode solver to find the modes of the structure
em.confinement(shape_list='core')   #calculate the confinement factor for each mode
em.report()                         #print information about the calculation results to command line
em.label_profile(name = 'dataset2') #store this set of results under label 'dataset2'
em.plot()

'ok'

In [6]:
print(wavelength_fund/2)

225.0


In [7]:
modenum_TM40 = 21   #check this in the plot output to confirm it's the right mode number for TM40 at the SHG wavelength
overlap = em.overlap(label_a='dataset2',mode_a=21,  label_b='dataset1', mode_b=0)
print('Power overlap TM40(SHG)+TM00(Fund): %0.10f %%' % (overlap*100))

modenum_TM04 = 30   #check this in the plot output to confirm it's the right mode number for TM04 at the SHG wavelength
overlap = em.overlap(label_a='dataset2',mode_a=modenum_TM04 ,  label_b='dataset1', mode_b=0)
print('Power overlap TM04(SHG)+TM00(Fund): %0.10f %%' % (overlap*100))



Power overlap TM40(SHG)+TM00(Fund): 0.0008477399 %
Power overlap TM04(SHG)+TM00(Fund): 0.0000000000 %


In [8]:
em.save(simulation_name='example_AlN_waveguide_overlap.eph')  # Save the simulation state to a fileem.save(simulation_name='example_AlN_waveguide_copy.eph')  # Save the simulation state to a file


TypeError: 'bool' object is not callable

In [ ]:
# LOOP over all SHG modes and calculate overlap with TM00 at the fundamental wavelength
overlap_results = []  #define a list
for mode_num_SHG in range(Nmodes_to_calculate):
    overlap = em.overlap(label_a='dataset2', mode_a=mode_num_SHG,  label_b='dataset1', mode_b=0)
    print('Power overlap SHG mode %d + TM00(Fund): %0.10f %%' % (mode_num_SHG, overlap*100))
    #save the overlap results to a file or data structure as needed
    overlap_results.append(overlap)

print(overlap_results)

#convert to a numeric array (numpy)
overlap_results_numeric = np.array(overlap_results)


In [ ]:
overlap_results_numeric = np.array(overlap_results)
print(overlap_results)

In [9]:
em.close()